# Task-01: Dataset Understanding & Preprocessing

This notebook explores the VisDrone dataset for the drone human and car detection task.

The goal of this step is to understand:
- dataset folder structure,
- annotation format,
- available object classes,
- target classes for this project,
- preprocessing and augmentation strategy,
- challenges in drone/aerial object detection,
- sample visualizations.

## Dataset Structure

The dataset is organized into train, validation, and test splits. Each split contains images and corresponding YOLO-format label files.

Expected structure:

```txt
VisDrone_Dataset/
├── VisDrone2019-DET-train/
│   ├── images/
│   └── labels/
├── VisDrone2019-DET-val/
│   ├── images/
│   └── labels/
├── VisDrone2019-DET-test-dev/
│   ├── images/
│   └── labels/
└── data.yaml

```


---

## Dataset Path Setup

```python

from pathlib import Path
import os

PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "VisDrone_Dataset"

TRAIN_IMG_DIR = DATA_ROOT / "VisDrone2019-DET-train" / "images"
TRAIN_LABEL_DIR = DATA_ROOT / "VisDrone2019-DET-train" / "labels"

VAL_IMG_DIR = DATA_ROOT / "VisDrone2019-DET-val" / "images"
VAL_LABEL_DIR = DATA_ROOT / "VisDrone2019-DET-val" / "labels"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATA_ROOT)
print("Train images exists:", TRAIN_IMG_DIR.exists())
print("Train labels exists:", TRAIN_LABEL_DIR.exists())
print("Validation images exists:", VAL_IMG_DIR.exists())
print("Validation labels exists:", VAL_LABEL_DIR.exists())
```

## Count Images and Label Files

```python
train_images = list(TRAIN_IMG_DIR.glob("*.jpg")) + list(TRAIN_IMG_DIR.glob("*.png"))
train_labels = list(TRAIN_LABEL_DIR.glob("*.txt"))

val_images = list(VAL_IMG_DIR.glob("*.jpg")) + list(VAL_IMG_DIR.glob("*.png"))
val_labels = list(VAL_LABEL_DIR.glob("*.txt"))

print("Training images:", len(train_images))
print("Training labels:", len(train_labels))
print("Validation images:", len(val_images))
print("Validation labels:", len(val_labels))
```

## Label Format

The labels are in YOLO format:

```txt
class_id x_center y_center width height
```

## Target classes

Human classes:
- pedestrian
- people

Vehicle class:
- car

total_humans = pedestrian_count + people_count

## Verify YOLO Annotation Format


```python
sample_label = train_labels[0]

print("Sample label file:", sample_label.name)

with open(sample_label, "r") as f:
    lines = f.readlines()[:10]

for line in lines:
    print(line.strip())

```

## Task-01 outputs

```python
%cd ..
!python src/explore_dataset.py
%cd notebooks
```

outputs/task01/class_distribution.png
outputs/task01/bbox_area_distribution.png
outputs/task01/sample_annotated_images.jpg

## Class Distribution Analysis

```python
from IPython.display import Image, display

display(Image(filename="../outputs/task01/class_distribution.png"))
```

The class distribution plot shows how many object annotations exist for each category.

This helps identify class imbalance. In drone datasets, some classes such as cars or pedestrians may appear more frequently than others. Since this project focuses only on humans and cars, I specifically check the number of pedestrian, people, and car annotations.

## Bounding Box Size Analysis

```python
display(Image(filename="../outputs/task01/bbox_area_distribution.png"))
```

The bounding box area distribution is important because drone images often contain very small objects.

Small humans and distant cars may occupy only a few pixels, making detection more difficult. This means the model needs to learn small object detection from aerial viewpoints.

## Sample Annotation Visualization

```python
display(Image(filename="../outputs/task01/sample_annotated_images.jpg"))
```

The sample visualization shows bounding boxes drawn on random training images.

Only the target classes for this assessment were visualized:
- pedestrian
- people
- car

This confirms that the labels are correctly connected with the images and that the annotation format can be used for training.

## Preprocessing Plan

The preprocessing steps for this project are:

1. Verify dataset structure  
   Check train, validation, and test folders.

2. Verify image-label pairs  
   Ensure each image has a corresponding label file.

3. Check annotation format  
   Confirm that labels follow YOLO format.

4. Select target classes  
   Use pedestrian and people as human classes, and car as the vehicle class.

5. Resize/letterbox images  
   During YOLO training, images will be resized to a fixed input size such as 640×640 while preserving aspect ratio.

6. Normalize image values  
   Pixel normalization will be handled by the YOLO training pipeline.

7. Visualize annotations  
   Draw bounding boxes on sample images to confirm label correctness.

## Augmentation Plan

The planned augmentations are:

1. Horizontal flip  
   Useful because drone scenes can appear from different directions.

2. Scaling and translation  
   Helps the model detect objects at different sizes and positions.

3. Brightness and contrast adjustment  
   Helps with outdoor lighting variation, shadows, and sunlight.

4. Mosaic augmentation  
   Helps YOLO learn from multiple scenes in one image and improves robustness.

5. Mild perspective transformation  
   Can help with aerial viewpoint variation, but should not be too aggressive because small objects may become distorted.

## Dataset Challenges

The main challenges noticed in this dataset are:

1. Small objects  
   Humans and cars can appear very small in drone images.

2. Dense scenes  
   Some images contain many objects close together, making detection and counting harder.

3. Occlusion  
   Objects can be partially hidden by trees, vehicles, buildings, or other people.

4. Similar vehicle classes  
   Cars, vans, trucks, and buses can look similar from aerial views.

5. Lighting variation  
   Outdoor scenes may contain shadows, bright sunlight, low contrast, or motion blur.

6. Class imbalance  
   Some classes may appear more frequently than others, which can affect model performance.